In [ ]:
!pip install tensorflow keras --quiet

import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.xception import Xception, preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

from google.colab import drive
drive.mount('/content/drive')

# --------------------------------------------
# CONFIG
# --------------------------------------------
DATASET_DIR = "/content/drive/MyDrive/deepfake_image_detection/dataset"
TRAIN_DIR = os.path.join(DATASET_DIR, "train")
TEST_DIR = os.path.join(DATASET_DIR, "test")

IMAGE_SIZE = (299, 299)
BATCH_SIZE = 16
INITIAL_EPOCHS = 10   # first stage: train top layers
FINETUNE_EPOCHS = 10  # second stage: unfreeze conv layers
LR = 3e-4

print("✔ Dataset path set:", DATASET_DIR)

# --------------------------------------------
# DATA PIPELINE
# --------------------------------------------
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=15,
    horizontal_flip=True,
    zoom_range=0.15,
    shear_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1
)

test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    class_mode="binary",
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    class_mode="binary",
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("✔ Data loaders created.")

# --------------------------------------------
# MODEL BUILDING (TRANSFER LEARNING)
# --------------------------------------------
base_model = Xception(weights="imagenet", include_top=False, input_shape=(299, 299, 3))

# ---- Stage 1: Freeze base model ----
base_model.trainable = False

x = GlobalAveragePooling2D()(base_model.output)
x = Dense(512, activation="relu")(x)
x = Dropout(0.4)(x)
output = Dense(1, activation="sigmoid")(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=Adam(LR),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# --------------------------------------------
# CALLBACKS
# --------------------------------------------
MODEL_SAVE_PATH = "/content/drive/MyDrive/deepfake_image_detection/models/xception_image_model.keras"

checkpoint = ModelCheckpoint(
    MODEL_SAVE_PATH,
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=3,
    min_lr=1e-6
)

callbacks = [checkpoint, early_stop, reduce_lr]

# --------------------------------------------
# TRAINING: Stage 1 (Train Top Layers Only)
# --------------------------------------------
history_1 = model.fit(
    train_generator,
    validation_data=test_generator,
    epochs=INITIAL_EPOCHS,
    callbacks=callbacks
)

# --------------------------------------------
# STAGE 2: FINE-TUNE (Unfreeze last 40% of layers)
# --------------------------------------------
for layer in base_model.layers[-70:]:  # unfreeze last block
    layer.trainable = True

model.compile(
    optimizer=Adam(1e-5),  # smaller LR for fine tuning
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("\n🔧 Fine-tuning model...\n")

history_2 = model.fit(
    train_generator,
    validation_data=test_generator,
    epochs=FINETUNE_EPOCHS,
    callbacks=callbacks
)

# --------------------------------------------
# PERFORMANCE PLOT
# --------------------------------------------
history_combined = {
    "accuracy": history_1.history["accuracy"] + history_2.history["accuracy"],
    "val_accuracy": history_1.history["val_accuracy"] + history_2.history["val_accuracy"],
    "loss": history_1.history["loss"] + history_2.history["loss"],
    "val_loss": history_1.history["val_loss"] + history_2.history["val_loss"]
}

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(history_combined["accuracy"], label="Train Acc")
plt.plot(history_combined["val_accuracy"], label="Val Acc")
plt.title("Accuracy")
plt.legend()

plt.subplot(1,2,2)
plt.plot(history_combined["loss"], label="Train Loss")
plt.plot(history_combined["val_loss"], label="Val Loss")
plt.title("Loss")
plt.legend()

plt.show()

# --------------------------------------------
# EVALUATION
# --------------------------------------------
loss, acc = model.evaluate(test_generator)
print(f"\n✔ Final Test Accuracy: {acc * 100:.2f}%")

print("✔ Model saved successfully to:", MODEL_SAVE_PATH)